In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Spotify Metadata Analysis") #this is the line im unsure of
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "18g")
    .config('spark.executor.instances', 7)
    .getOrCreate()
)
#double check

In [16]:
#spotify_clean_parquet
#df = spark.read.parquet("/expanse/lustre/projects/uci157/zlenus/shared/spotify_clean_parquet")
tracks = spark.read.parquet("/expanse/lustre/projects/uci157/zlenus/shared/spotify_clean_parquet/tracks.parquet")
available_markets= spark.read.parquet("/expanse/lustre/projects/uci157/zlenus/shared/spotify_clean_parquet/available_markets.parquet")
track_artists = spark.read.parquet("/expanse/lustre/projects/uci157/zlenus/shared/spotify_clean_parquet/track_artists.parquet")
artists = spark.read.parquet("/expanse/lustre/projects/uci157/zlenus/shared/spotify_clean_parquet/artists.parquet")
artist_genres = spark.read.parquet("/expanse/lustre/projects/uci157/zlenus/shared/spotify_clean_parquet/artist_genres.parquet")
artist_albums = spark.read.parquet("/expanse/lustre/projects/uci157/zlenus/shared/spotify_clean_parquet/artist_albums.parquet")
albums = spark.read.parquet("/expanse/lustre/projects/uci157/zlenus/shared/spotify_clean_parquet/albums.parquet")


In [20]:
tracks.count() + available_markets.count() + track_artists.count() + artists.count() + artist_genres.count() + artist_albums.count() + albums.count()

792244463

In [22]:
tracks.printSchema()

root
 |-- rowid: long (nullable = true)
 |-- id: string (nullable = true)
 |-- fetched_at: long (nullable = true)
 |-- name: string (nullable = true)
 |-- preview_url: string (nullable = true)
 |-- album_rowid: long (nullable = true)
 |-- track_number: long (nullable = true)
 |-- external_id_isrc: string (nullable = true)
 |-- popularity: long (nullable = true)
 |-- available_markets_rowid: long (nullable = true)
 |-- disc_number: long (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- explicit: long (nullable = true)



In [23]:
available_markets.printSchema()

root
 |-- rowid: long (nullable = true)
 |-- available_markets: string (nullable = true)



In [24]:
track_artists.printSchema()

root
 |-- track_rowid: long (nullable = true)
 |-- artist_rowid: long (nullable = true)



In [25]:
artists.printSchema()

root
 |-- rowid: long (nullable = true)
 |-- id: string (nullable = true)
 |-- fetched_at: long (nullable = true)
 |-- name: string (nullable = true)
 |-- followers_total: long (nullable = true)
 |-- popularity: long (nullable = true)



In [26]:
artist_genres.printSchema()

root
 |-- artist_rowid: long (nullable = true)
 |-- genre: string (nullable = true)



In [27]:
artist_albums.printSchema()

root
 |-- artist_rowid: long (nullable = true)
 |-- album_rowid: long (nullable = true)
 |-- is_appears_on: long (nullable = true)
 |-- is_implicit_appears_on: long (nullable = true)
 |-- index_in_album: long (nullable = true)



In [28]:
albums.printSchema()

root
 |-- rowid: long (nullable = true)
 |-- id: string (nullable = true)
 |-- fetched_at: long (nullable = true)
 |-- name: string (nullable = true)
 |-- album_type: string (nullable = true)
 |-- available_markets_rowid: long (nullable = true)
 |-- external_id_upc: string (nullable = true)
 |-- copyright_c: string (nullable = true)
 |-- copyright_p: string (nullable = true)
 |-- label: string (nullable = true)
 |-- popularity: long (nullable = true)
 |-- release_date: string (nullable = true)
 |-- release_date_precision: string (nullable = true)
 |-- total_tracks: long (nullable = true)
 |-- external_id_amgid: string (nullable = true)



In [30]:
from pyspark.sql.functions import col, count, when

null_counts = tracks.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in tracks.columns
])

null_counts.show()

+-----+---+----------+----+-----------+-----------+------------+----------------+----------+-----------------------+-----------+-----------+--------+
|rowid| id|fetched_at|name|preview_url|album_rowid|track_number|external_id_isrc|popularity|available_markets_rowid|disc_number|duration_ms|explicit|
+-----+---+----------+----+-----------+-----------+------------+----------------+----------+-----------------------+-----------+-----------+--------+
|    0|  0|         0|   0|   25410128|          0|           0|          104792|         0|                      0|          0|          0|       0|
+-----+---+----------+----+-----------+-----------+------------+----------------+----------+-----------------------+-----------+-----------+--------+



In [35]:
null_counts = available_markets.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in available_markets.columns
])

null_counts.show()

+-----+-----------------+
|rowid|available_markets|
+-----+-----------------+
|    0|                0|
+-----+-----------------+



In [37]:
null_counts = track_artists.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in track_artists.columns
])

null_counts.show()

+-----------+------------+
|track_rowid|artist_rowid|
+-----------+------------+
|          0|           0|
+-----------+------------+



In [38]:
null_counts = artists.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in artists.columns
])

null_counts.show()

+-----+---+----------+----+---------------+----------+
|rowid| id|fetched_at|name|followers_total|popularity|
+-----+---+----------+----+---------------+----------+
|    0|  0|         0|   0|              0|         0|
+-----+---+----------+----+---------------+----------+



In [41]:
null_counts = artist_genres.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in artist_genres.columns
])

null_counts.show()

+------------+-----+
|artist_rowid|genre|
+------------+-----+
|           0|    0|
+------------+-----+



In [42]:
null_counts = artist_albums.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in artist_albums.columns
])

null_counts.show()

+------------+-----------+-------------+----------------------+--------------+
|artist_rowid|album_rowid|is_appears_on|is_implicit_appears_on|index_in_album|
+------------+-----------+-------------+----------------------+--------------+
|           0|          0|            0|                     0|      45093900|
+------------+-----------+-------------+----------------------+--------------+



In [43]:
null_counts = albums.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in albums.columns
])

null_counts.show()

+-----+---+----------+----+----------+-----------------------+---------------+-----------+-----------+-----+----------+------------+----------------------+------------+-----------------+
|rowid| id|fetched_at|name|album_type|available_markets_rowid|external_id_upc|copyright_c|copyright_p|label|popularity|release_date|release_date_precision|total_tracks|external_id_amgid|
+-----+---+----------+----+----------+-----------------------+---------------+-----------+-----------+-----+----------+------------+----------------------+------------+-----------------+
|    0|  0|         0|   0|         0|                      0|          48056|     907818|    1195597|    0|         0|           0|                     0|           0|         58574587|
+-----+---+----------+----+----------+-----------------------+---------------+-----------+-----------+-----+----------+------------+----------------------+------------+-----------------+

